## 1. Imports y carga inicial

In [0]:
from pyspark.sql import functions as f
from pyspark.sql import Window

BASE = "/Volumes/mine4213/proyecto/data"
CSV_DIR = f"{BASE}/csv"

AIS_SCHEMA = """
    MMSI string,
    BaseDateTime timestamp,
    LAT double,
    LON double,
    SOG float,
    COG float,
    Heading float,
    VesselName string,
    IMO string,
    CallSign string,
    VesselType smallint,
    Status smallint,
    Length float,
    Width float,
    Draft float,
    Cargo string,
    TransceiverClass string
"""

ais = (
    spark.read
    .option("header", True)
    .option("timestampFormat", "yyyy-MM-dd'T'HH:mm:ss")
    .schema(AIS_SCHEMA)
    .csv(f"{CSV_DIR}/AIS_2023_06_*.csv")
    .withColumn("fecha", f.to_date("BaseDateTime"))
    .withColumn(
        "archivo_origen",
        f.regexp_extract(
            f.col("_metadata.file_name"),
            r"(AIS_2023_06_\d{2}\.csv)",
            1
        )
    )
    .withColumn(
        "fecha_archivo",
        f.to_date(
            f.regexp_extract(
                f.col("_metadata.file_name"),
                r"AIS_(\d{4}_\d{2}_\d{2})\.csv",
                1
            ),
            "yyyy_MM_dd"
        )
    )
)

ais.printSchema()

root
 |-- MMSI: string (nullable = true)
 |-- BaseDateTime: timestamp (nullable = true)
 |-- LAT: double (nullable = true)
 |-- LON: double (nullable = true)
 |-- SOG: float (nullable = true)
 |-- COG: float (nullable = true)
 |-- Heading: float (nullable = true)
 |-- VesselName: string (nullable = true)
 |-- IMO: string (nullable = true)
 |-- CallSign: string (nullable = true)
 |-- VesselType: short (nullable = true)
 |-- Status: short (nullable = true)
 |-- Length: float (nullable = true)
 |-- Width: float (nullable = true)
 |-- Draft: float (nullable = true)
 |-- Cargo: string (nullable = true)
 |-- TransceiverClass: string (nullable = true)
 |-- fecha: date (nullable = true)
 |-- archivo_origen: string (nullable = false)
 |-- fecha_archivo: date (nullable = false)



## 2. Posiciones y buques unicos por dia

In [0]:
perfil_diario = (
    ais
    .groupBy("fecha")
    .agg(
        f.count("*").alias("posiciones"),
        f.countDistinct("MMSI").alias("buques_unicos")
    )
    .orderBy("fecha")
)

display(perfil_diario)

fecha,posiciones,buques_unicos
2023-06-01,8808904,20448
2023-06-02,9052241,21153
2023-06-03,8036348,20505
2023-06-04,8522645,19720
2023-06-05,8613757,19615
2023-06-06,8587938,19717
2023-06-07,8911726,20086


Databricks visualization. Run in Databricks to view.

### Total semanal

In [0]:
resumen_general = (
    ais
    .agg(
        f.count("*").alias("total_posiciones"),
        f.countDistinct("MMSI").alias("buques_unicos_semana"),
        f.min("BaseDateTime").alias("primer_timestamp"),
        f.max("BaseDateTime").alias("ultimo_timestamp")
    )
)

display(resumen_general)

total_posiciones,buques_unicos_semana,primer_timestamp,ultimo_timestamp
60533559,31871,2023-06-01T00:00:00.000Z,2023-06-07T23:59:59.000Z


## 3. Completitud

In [0]:
columnas_originales = [
    "MMSI",
    "BaseDateTime",
    "LAT",
    "LON",
    "SOG",
    "COG",
    "Heading",
    "VesselName",
    "IMO",
    "CallSign",
    "VesselType",
    "Status",
    "Length",
    "Width",
    "Draft",
    "Cargo",
    "TransceiverClass"
]

string_cols = {
    field.name
    for field in ais.schema.fields
    if field.dataType.simpleString() == "string"
}

missing_exprs = []

for col_name in columnas_originales:

    if col_name in string_cols:
        missing_condition = (
            f.col(col_name).isNull()
            | (f.trim(f.col(col_name)) == "")
        )
    else:
        missing_condition = f.col(col_name).isNull()

    missing_exprs.append(
        f.sum(
            f.when(missing_condition, 1).otherwise(0)
        ).alias(col_name)
    )

missing_row = (
    ais
    .agg(
        f.count("*").alias("total_filas"),
        *missing_exprs
    )
    .first()
)

total_filas = missing_row["total_filas"]

perfil_completitud = spark.createDataFrame(
    [
        (
            col_name,
            int(missing_row[col_name]),
            round(
                100 * missing_row[col_name] / total_filas,
                4
            )
        )
        for col_name in columnas_originales
    ],
    [
        "columna",
        "valores_faltantes",
        "porcentaje_faltante"
    ]
)

display(
    perfil_completitud
    .orderBy(f.desc("porcentaje_faltante"))
)

columna,valores_faltantes,porcentaje_faltante
Draft,39000865,64.4285
IMO,25891448,42.7721
Status,19952397,32.9609
Cargo,19898489,32.8718
CallSign,10157915,16.7806
Width,9219567,15.2305
Length,3664725,6.054
VesselType,209863,0.3467
VesselName,150685,0.2489
MMSI,0,0.0


## 4. Distribucion por tipo de buqeu

In [0]:
w_total = Window.partitionBy()

distribucion_tipo = (
    ais
    .groupBy("VesselType")
    .agg(
        f.count("*").alias("posiciones"),
        f.countDistinct("MMSI").alias("buques_unicos")
    )
    .withColumn(
        "porcentaje_posiciones",
        f.round(
            100
            * f.col("posiciones")
            / f.sum("posiciones").over(w_total),
            4
        )
    )
    .orderBy(f.desc("posiciones"))
)

display(distribucion_tipo)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


VesselType,posiciones,buques_unicos,porcentaje_posiciones
31,16558120,3451,27.3536
37,14476696,12934,23.9152
60,4795209,1708,7.9216
30,4010188,2192,6.6247
36,3876264,4243,6.4035
90,3848162,1532,6.3571
70,3837926,1563,6.3402
52,1921565,482,3.1744
80,1789168,818,2.9557
57,1323765,242,2.1868


In [0]:
display(
    distribucion_tipo
    .filter(f.col("VesselType").isNull())
)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


VesselType,posiciones,buques_unicos,porcentaje_posiciones
null,209863,851,0.3467


In [0]:
display(distribucion_tipo.limit(20))

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


VesselType,posiciones,buques_unicos,porcentaje_posiciones
31,16558120,3451,27.3536
37,14476696,12934,23.9152
60,4795209,1708,7.9216
30,4010188,2192,6.6247
36,3876264,4243,6.4035
90,3848162,1532,6.3571
70,3837926,1563,6.3402
52,1921565,482,3.1744
80,1789168,818,2.9557
57,1323765,242,2.1868


## 5. Distribucion por tama;o

In [0]:
dimensiones_validas = (
    ais
    .select(
        f.when(f.col("Length") > 0, f.col("Length")).alias("Length"),
        f.when(f.col("Width") > 0, f.col("Width")).alias("Width"),
        f.when(f.col("Draft") > 0, f.col("Draft")).alias("Draft")
    )
)

display(
    dimensiones_validas.summary(
        "count",
        "mean",
        "stddev",
        "min",
        "25%",
        "50%",
        "75%",
        "90%",
        "95%",
        "99%",
        "max"
    )
)

summary,Length,Width,Draft
count,54789691,49086707,18813232
mean,50.4669292987982,11.255713038562558,5.738716648594378
stddev,68.56868482038146,9.767056861472252,3.8808895834591373
min,1.0,1.0,0.1
25%,16.0,6.0,3.0
50%,23.0,8.0,4.2
75%,40.0,12.0,8.2
90%,177.0,27.0,11.9
95%,215.0,32.0,13.5
99%,325.0,48.0,17.0


In [0]:
ais_tamano = (
    ais
    .withColumn(
        "categoria_tamano",
        f.when(
            f.col("Length").isNull() | (f.col("Length") <= 0),
            "Sin longitud utilizable"
        )
        .when(f.col("Length") < 25, "< 25 m")
        .when(f.col("Length") < 50, "25 - 49.9 m")
        .when(f.col("Length") < 100, "50 - 99.9 m")
        .when(f.col("Length") < 200, "100 - 199.9 m")
        .otherwise(">= 200 m")
    )
)

distribucion_tamano = (
    ais_tamano
    .groupBy("categoria_tamano")
    .agg(
        f.count("*").alias("posiciones"),
        f.countDistinct("MMSI").alias("buques_unicos")
    )
    .withColumn(
        "porcentaje_posiciones",
        f.round(
            100
            * f.col("posiciones")
            / f.sum("posiciones").over(w_total),
            4
        )
    )
    .orderBy(f.desc("posiciones"))
)

display(distribucion_tamano)

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1155: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


categoria_tamano,posiciones,buques_unicos,porcentaje_posiciones
< 25 m,29782752,19403,49.2004
25 - 49.9 m,13947421,3880,23.0408
Sin longitud utilizable,5743868,4577,9.4887
100 - 199.9 m,4316156,1693,7.1302
50 - 99.9 m,3376748,1015,5.5783
>= 200 m,3366614,1303,5.5616


In [0]:
tipo_tamano = (
    ais_tamano
    .groupBy(
        "VesselType",
        "categoria_tamano"
    )
    .agg(
        f.count("*").alias("posiciones"),
        f.countDistinct("MMSI").alias("buques_unicos")
    )
    .orderBy(f.desc("posiciones"))
)

display(tipo_tamano.limit(50))

VesselType,categoria_tamano,posiciones,buques_unicos
37,< 25 m,10328616,9892
31,< 25 m,9089040,1845
31,25 - 49.9 m,6478902,1359
36,< 25 m,3472397,3846
37,Sin longitud utilizable,2312536,2191
30,< 25 m,2128678,1415
60,25 - 49.9 m,1890879,548
70,100 - 199.9 m,1826745,811
37,25 - 49.9 m,1592684,742
70,>= 200 m,1568272,579


## 6. Reglas de caldiad

In [0]:
inicio_semana = f.lit("2023-06-01 00:00:00").cast("timestamp")
fin_semana = f.lit("2023-06-08 00:00:00").cast("timestamp")

checks = [
    (
        "coordenadas_nulas",
        f.col("LAT").isNull() | f.col("LON").isNull(),
        "LAT o LON no disponibles"
    ),
    (
        "lat_fuera_rango",
        (f.col("LAT") < -90) | (f.col("LAT") > 90),
        "Latitud fuera de [-90, 90]"
    ),
    (
        "lon_fuera_rango",
        (f.col("LON") < -180) | (f.col("LON") > 180),
        "Longitud fuera de [-180, 180]"
    ),

    (
        "sog_nulo",
        f.col("SOG").isNull(),
        "Velocidad no informada"
    ),
    (
        "sog_no_disponible_102_3",
        f.abs(f.col("SOG") - f.lit(102.3)) < 0.001,
        "102.3 es el valor AIS de SOG no disponible"
    ),
    (
        "sog_fuera_codificacion_ais",
        (f.col("SOG") < 0) | (f.col("SOG") > 102.3),
        "Valor fuera del rango esperado para SOG AIS"
    ),
    (
        "sog_mayor_60_sospechosa",
        (f.col("SOG") > 60) & (f.col("SOG") <= 102.2),
        "Velocidad muy alta: revisar, no eliminar automáticamente"
    ),

    (
        "cog_no_disponible_360",
        f.abs(f.col("COG") - f.lit(360.0)) < 0.001,
        "COG=360 significa no disponible"
    ),
    (
        "cog_fuera_rango",
        (f.col("COG") < 0) | (f.col("COG") > 360),
        "COG fuera del rango AIS"
    ),

    (
        "heading_no_disponible_511",
        f.abs(f.col("Heading") - f.lit(511.0)) < 0.001,
        "Heading=511 significa no disponible"
    ),
    (
        "heading_fuera_rango",
        (f.col("Heading") < 0)
        | (
            (f.col("Heading") > 359)
            & (f.abs(f.col("Heading") - f.lit(511.0)) >= 0.001)
        ),
        "Heading distinto de 0-359 y no es el sentinel 511"
    ),

    (
        "mmsi_nulo_o_vacio",
        f.col("MMSI").isNull()
        | (f.trim(f.col("MMSI")) == ""),
        "MMSI ausente"
    ),
    (
        "mmsi_formato_invalido",
        f.col("MMSI").isNotNull()
        & (~f.trim(f.col("MMSI")).rlike(r"^[0-9]{9}$")),
        "MMSI que no contiene exactamente 9 dígitos"
    ),

    (
        "timestamp_nulo",
        f.col("BaseDateTime").isNull(),
        "Timestamp ausente"
    ),
    (
        "timestamp_fuera_semana",
        (f.col("BaseDateTime") < inicio_semana)
        | (f.col("BaseDateTime") >= fin_semana),
        "Timestamp fuera del corpus 1-7 junio"
    ),
    (
        "fecha_no_coincide_archivo",
        f.col("fecha").isNotNull()
        & f.col("fecha_archivo").isNotNull()
        & (f.col("fecha") != f.col("fecha_archivo")),
        "El timestamp no corresponde al día indicado por el archivo"
    ),

    (
        "length_negativo",
        f.col("Length") < 0,
        "Longitud físicamente inválida"
    ),
    (
        "width_negativo",
        f.col("Width") < 0,
        "Ancho físicamente inválido"
    ),
    (
        "draft_negativo",
        f.col("Draft") < 0,
        "Calado físicamente inválido"
    ),

    (
        "length_cero",
        f.col("Length") == 0,
        "Longitud cero; posiblemente no disponible"
    ),
    (
        "width_cero",
        f.col("Width") == 0,
        "Ancho cero; posiblemente no disponible"
    ),
    (
        "draft_cero",
        f.col("Draft") == 0,
        "Calado cero; posiblemente no disponible"
    ),

    (
        "vessel_type_nulo",
        f.col("VesselType").isNull(),
        "Tipo de buque no informado"
    )
]

In [0]:
quality_exprs = []

for i, (_, condition, _) in enumerate(checks):
    quality_exprs.append(
        f.sum(
            f.when(condition, 1).otherwise(0)
        ).alias(f"q_{i}")
    )

quality_row = (
    ais
    .agg(
        f.count("*").alias("total_filas"),
        *quality_exprs
    )
    .first()
)

total_filas = quality_row["total_filas"]

In [0]:
quality_data = []

for i, (nombre, _, descripcion) in enumerate(checks):

    cantidad = int(quality_row[f"q_{i}"])

    porcentaje = round(
        100 * cantidad / total_filas,
        6
    )

    quality_data.append(
        (
            nombre,
            cantidad,
            porcentaje,
            descripcion
        )
    )

diagnostico_calidad = spark.createDataFrame(
    quality_data,
    [
        "regla",
        "filas_afectadas",
        "porcentaje",
        "interpretacion"
    ]
)

display(
    diagnostico_calidad
    .orderBy(f.desc("filas_afectadas"))
)

regla,filas_afectadas,porcentaje,interpretacion
heading_no_disponible_511,33535628,55.40006,Heading=511 significa no disponible
cog_no_disponible_360,10291131,17.000704,COG=360 significa no disponible
draft_cero,2719462,4.492487,Calado cero; posiblemente no disponible
width_cero,2227285,3.679422,Ancho cero; posiblemente no disponible
length_cero,2079143,3.434695,Longitud cero; posiblemente no disponible
vessel_type_nulo,209863,0.346689,Tipo de buque no informado
sog_no_disponible_102_3,159987,0.264295,102.3 es el valor AIS de SOG no disponible
sog_fuera_codificacion_ais,159987,0.264295,Valor fuera del rango esperado para SOG AIS
mmsi_formato_invalido,49897,0.082429,MMSI que no contiene exactamente 9 dígitos
heading_fuera_rango,1061,0.001753,Heading distinto de 0-359 y no es el sentinel 511


## 7. Velocidades altas

In [0]:
display(
    ais
    .filter(
        (f.col("SOG") > 60)
        & (f.col("SOG") <= 102.2)
    )
    .select(
        "MMSI",
        "BaseDateTime",
        "LAT",
        "LON",
        "SOG",
        "VesselType",
        "VesselName"
    )
    .orderBy(f.desc("SOG"))
    .limit(100)
)

MMSI,BaseDateTime,LAT,LON,SOG,VesselType,VesselName
338457411,2023-06-01T10:12:28.000Z,46.80104,-108.0056,102.2,37,MAS PESOS
367593530,2023-06-01T10:16:55.000Z,27.34855,-120.5486,102.2,37,SEA WEED
338457411,2023-06-01T10:40:27.000Z,48.69449,-110.22188,102.2,37,MAS PESOS
338457411,2023-06-01T10:44:31.000Z,48.36764,-111.30339,102.2,37,MAS PESOS
338457411,2023-06-01T10:26:00.000Z,47.59117,-108.60642,102.2,37,MAS PESOS
338457411,2023-06-01T09:55:28.000Z,46.12951,-107.81249,102.2,37,MAS PESOS
338457411,2023-06-01T10:33:28.000Z,48.12831,-109.26436,102.2,37,MAS PESOS
982470018,2023-06-01T10:39:23.000Z,46.00493,-122.58101,102.2,90,COSTA LUMINOSA RB 1
338199886,2023-06-07T07:16:20.000Z,30.03227,-83.89173,102.2,37,THRESHER
338457411,2023-06-01T10:37:27.000Z,48.4448,-109.76057,102.2,37,MAS PESOS


In [0]:
display(
    ais
    .select("SOG")
    .filter(f.col("SOG").isNotNull())
    .summary(
        "count",
        "mean",
        "stddev",
        "min",
        "50%",
        "90%",
        "95%",
        "99%",
        "max"
    )
)

summary,SOG
count,60533559
mean,2.6289088393857143
stddev,6.922652105468487
min,0.0
50%,0.0
90%,9.0
95%,12.5
99%,22.0
max,102.3


## 8. Duplicados
Duplciado = mismo MMSI + BaseDateTime

In [0]:
duplicados_timestamp = (
    ais
    .filter(
        f.col("MMSI").isNotNull()
        & f.col("BaseDateTime").isNotNull()
    )
    .groupBy(
        "MMSI",
        "BaseDateTime"
    )
    .agg(
        f.count("*").alias("numero_registros"),
        f.countDistinct(
            f.struct("LAT", "LON")
        ).alias("posiciones_distintas")
    )
    .filter(
        f.col("numero_registros") > 1
    )
)

In [0]:
resumen_duplicados = (
    duplicados_timestamp
    .agg(
        f.count("*").alias(
            "claves_mmsi_timestamp_duplicadas"
        ),

        f.sum("numero_registros").alias(
            "filas_en_claves_duplicadas"
        ),

        f.sum(
            f.col("numero_registros") - 1
        ).alias(
            "filas_excedentes"
        ),

        f.sum(
            f.when(
                f.col("posiciones_distintas") == 1,
                1
            ).otherwise(0)
        ).alias(
            "claves_con_misma_posicion"
        ),

        f.sum(
            f.when(
                f.col("posiciones_distintas") > 1,
                1
            ).otherwise(0)
        ).alias(
            "claves_con_posiciones_conflictivas"
        )
    )
)

display(resumen_duplicados)

claves_mmsi_timestamp_duplicadas,filas_en_claves_duplicadas,filas_excedentes,claves_con_misma_posicion,claves_con_posiciones_conflictivas
1672,3344,1672,1400,272


In [0]:
display(
    duplicados_timestamp
    .orderBy(
        f.desc("numero_registros")
    )
    .limit(100)
)

MMSI,BaseDateTime,numero_registros,posiciones_distintas
255805854,2023-06-03T12:00:01.000Z,2,2
366772960,2023-06-02T21:00:08.000Z,2,2
367394780,2023-06-05T18:00:00.000Z,2,1
367503760,2023-06-05T00:00:00.000Z,2,1
354162000,2023-06-04T12:59:59.000Z,2,1
311000438,2023-06-06T22:00:00.000Z,2,1
367480010,2023-06-04T10:59:59.000Z,2,1
367608860,2023-06-06T13:56:09.000Z,2,2
368093210,2023-06-05T04:59:59.000Z,2,1
366943250,2023-06-04T05:00:00.000Z,2,1


In [0]:
display(
    duplicados_timestamp
    .filter(
        f.col("posiciones_distintas") > 1
    )
    .orderBy(
        f.desc("numero_registros")
    )
    .limit(100)
)

MMSI,BaseDateTime,numero_registros,posiciones_distintas
477110600,2023-06-03T21:49:41.000Z,2,2
566013000,2023-06-04T16:29:29.000Z,2,2
440141000,2023-06-03T23:29:01.000Z,2,2
352633000,2023-06-04T20:08:49.000Z,2,2
367533290,2023-06-04T19:16:08.000Z,2,2
369285000,2023-06-02T21:00:07.000Z,2,2
566071000,2023-06-03T20:00:00.000Z,2,2
367308610,2023-06-05T02:59:59.000Z,2,2
338070397,2023-06-03T20:53:11.000Z,2,2
338097196,2023-06-04T22:54:29.000Z,2,2


## 9. MMSI anomalso

In [0]:
mmsi_anomalos = (
    ais
    .filter(
        f.col("MMSI").isNull()
        | (f.trim(f.col("MMSI")) == "")
        | (~f.trim(f.col("MMSI")).rlike(r"^[0-9]{9}$"))
    )
    .groupBy("MMSI")
    .agg(
        f.count("*").alias("posiciones")
    )
    .orderBy(
        f.desc("posiciones")
    )
)

display(mmsi_anomalos)

MMSI,posiciones
3669884,8623
69918000,8273
3669883,4973
9110192,3773
63280801,3721
1072211352,2838
2981331,2759
1,2516
3126547,1827
3660489,1105


## 10. Coordenadas anomalas

In [0]:
display(
    ais
    .filter(
        (f.col("LAT") < -90)
        | (f.col("LAT") > 90)
        | (f.col("LON") < -180)
        | (f.col("LON") > 180)
    )
    .groupBy(
        "LAT",
        "LON"
    )
    .agg(
        f.count("*").alias("apariciones")
    )
    .orderBy(
        f.desc("apariciones")
    )
)

LAT,LON,apariciones


## 11. Problemas de calidad por dia

In [0]:
calidad_por_dia = (
    ais
    .groupBy("fecha")
    .agg(
        f.count("*").alias("posiciones"),

        f.sum(
            f.when(
                (f.col("LAT") < -90)
                | (f.col("LAT") > 90)
                | (f.col("LON") < -180)
                | (f.col("LON") > 180),
                1
            ).otherwise(0)
        ).alias("coordenadas_fuera_rango"),

        f.sum(
            f.when(
                f.abs(f.col("SOG") - 102.3) < 0.001,
                1
            ).otherwise(0)
        ).alias("sog_no_disponible"),

        f.sum(
            f.when(
                f.col("MMSI").isNull()
                | (~f.trim(f.col("MMSI")).rlike(r"^[0-9]{9}$")),
                1
            ).otherwise(0)
        ).alias("mmsi_anomalo")
    )
    .orderBy("fecha")
)

display(calidad_por_dia)

fecha,posiciones,coordenadas_fuera_rango,sog_no_disponible,mmsi_anomalo
2023-06-01,8808904,0,21946,6911
2023-06-02,9052241,0,22531,7876
2023-06-03,8036348,0,21033,7173
2023-06-04,8522645,0,22133,6308
2023-06-05,8613757,0,23253,6502
2023-06-06,8587938,0,24426,7055
2023-06-07,8911726,0,24665,8072
